In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.ipynb").exists())
get_ipython().run_line_magic("run", '-i "' + (ROOT / "src" / "common.ipynb").as_posix() + '"')


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.ticker import FormatStrFormatter

FIELDS = FEATURES
predictors_mapping = LABELS
SEASON_CFG = {code: {} for code in SEASONS}
PANEL_LABELS = ["a", "b", "c", "d"]
TOP_K = 13
MAP_CRS = "EPSG:2163"
COUNTY_EDGE_COLOR = "white"
COUNTY_EDGE_WIDTH = 0.001
STATE_EDGE_COLOR = "#363535"
STATE_EDGE_WIDTH = 0.35
OTHERS_COLOR = "#4A86E8"
NO_DATA_COLOR = "#E6E6E6"
FORMATS = ["png", "pdf"]
plot_folder = OUT / "figures"
plot_folder.mkdir(parents=True, exist_ok=True)
FIGURE_NAMES = {
    "seasonal_rank1": "Figure6", "overall_top3": "FigureS4",
    "seasonal_mean_rank1": "FigureS5", "seasonal_rank2": "FigureS6",
    "seasonal_mean_rank2": "FigureS7", "seasonal_rank3": "FigureS8",
    "seasonal_mean_rank3": "FigureS9"
}
FULL_FEATURE_COLOR_MAP = {'LATITUDE': '#7FC97F', 'LONGITUDE': '#FFED6F', 'surface_net_thermal_radiation_sum': '#C56280', 'wind_speed_10m': '#9C755F', 'PM25': '#165004D7', 'NTL': '#F39C12', 'NDBI': '#CCEBC5', 'EVI': '#E7298A', 'ACS_PCT_BLACK_NONHISP': '#66A61E', 'ACS_PCT_HISPANIC': '#E6AB02', 'ACS_PCT_AGE_ABOVE65': '#A6761D', 'ACS_PCT_ENGL_NOT_ALL': '#7570B3', 'ACS_PCT_PERSON_INC_BELOW99': '#1B9E77', 'ACS_PCT_UNEMPLOY': '#B3DE69', 'PCT_DEM': '#6453C2', 'AC_RECS': '#80B1D3', 'ACS_PCT_HU_ELEC': '#FB8072', 'ACS_PCT_OWNER_HU_COST_30PCT': '#B752B5', 'ACS_PCT_RENTER_HU': '#FDB462', 'ACS_PCT_1UP_PERS_1ROOM': '#8DD3C7', 'ACS_MEDIAN_HOME_VALUE': '#BC80BD', 'ACS_MEDIAN_YEAR_BUILT': '#FCCDE5', 'HDD': '#BEBADA', 'CDD': '#6AA84F', 'RH_mean': '#19B5D8', 'ACS_PCT_HU_SOLAR': '#D95F02'}
plt.rcdefaults()
plt.rcParams.update({"svg.fonttype": "none", "pdf.fonttype": 42, "ps.fonttype": 42})

def save_plot(fig, stem):
    for extension in FORMATS:
        kwargs = {"bbox_inches": "tight"}
        if extension == "png":
            kwargs["dpi"] = 900
        if extension == "pdf":
            kwargs["dpi"] = 900
            if stem == "FigureS4":
                kwargs["transparent"] = True
            for axis in fig.axes:
                for collection in axis.collections:
                    collection.set_rasterized(True)
        fig.savefig(plot_folder / f"{stem}.{extension}", **kwargs)
        if extension == "pdf":
            for axis in fig.axes:
                for collection in axis.collections:
                    collection.set_rasterized(False)
    plt.close(fig)


In [ ]:
def pretty_driver_name(fid):
    if pd.isna(fid):
        return np.nan
    return predictors_mapping[fid]

def compute_ranks(frame):
    out = frame.copy()
    cols = [f'Imp_{f}' for f in FIELDS]
    arr = out[cols].to_numpy(dtype=float)
    zero_gain = np.all(arr == 0, axis=1)
    out['__zero_gain_model__'] = zero_gain
    valid_rows = np.flatnonzero(~zero_gain)
    order = np.argsort(arr[valid_rows], axis=1)[:, ::-1]
    names = np.asarray(FIELDS)
    for rank in [1, 2, 3]:
        ids = np.full(len(out), np.nan, dtype=object)
        importance = np.full(len(out), np.nan, dtype=float)
        ids[valid_rows] = names[order[:, rank - 1]]
        importance[valid_rows] = arr[valid_rows, order[:, rank - 1]]
        out[f'Top{rank}FeatureID'] = ids
        out[f'Top{rank}Importance'] = importance
        out[f'Top{rank}FeatureName'] = out[f'Top{rank}FeatureID'].map(predictors_mapping)
    return out

def season_level_top_raw_feature_ids(gdf, top_n=3):
    cols = [f'Imp_{f}' for f in FIELDS]
    valid = gdf
    if '__zero_gain_model__' in gdf:
        valid = gdf.loc[~gdf['__zero_gain_model__'].eq(True)]
    means = valid[cols].mean(axis=0, skipna=True).dropna().sort_values(ascending=False)
    return [name[4:] for name in means.index[:top_n]]

def overall_top_raw_feature_ids(gdf, top_n=3):
    return season_level_top_raw_feature_ids(gdf, top_n)

def get_rank_top8_feature_ids(season_gdfs, rank=1, top_k=8):
    rank_col = f'Top{rank}FeatureID'
    all_ids = []
    for season in SEASON_CFG.keys():
        vals = pd.Series(season_gdfs[season][rank_col]).dropna()
        vals = vals[vals.isin(FULL_FEATURE_COLOR_MAP.keys())]
        all_ids.append(vals)
    if len(all_ids) == 0:
        return []
    all_ids = pd.concat(all_ids, ignore_index=True)
    if all_ids.empty:
        return []
    return all_ids.value_counts().index.tolist()[:top_k]

def apply_top8_fixed_colors(gdf, feature_col, top_ids):
    out = gdf.copy()
    out['FeaturePlotID'] = np.where(out[feature_col].isna(), np.nan, np.where(out[feature_col].isin(top_ids), out[feature_col], 'OTHERS'))
    color_map = {fid: FULL_FEATURE_COLOR_MAP[fid] for fid in top_ids if fid in FULL_FEATURE_COLOR_MAP}
    plot_colors = out['FeaturePlotID'].map(color_map)
    plot_colors = np.where(out['FeaturePlotID'] == 'OTHERS', OTHERS_COLOR, plot_colors)
    plot_colors = np.where(pd.isna(out['FeaturePlotID']), NO_DATA_COLOR, plot_colors)
    return (out, plot_colors, color_map)

def build_top8_legend_handles(top_ids, include_others=True, include_nodata=True):
    handles = [mpatches.Patch(color=FULL_FEATURE_COLOR_MAP[fid], label=pretty_driver_name(fid)) for fid in top_ids if fid in FULL_FEATURE_COLOR_MAP]
    if include_others:
        handles.append(mpatches.Patch(color=OTHERS_COLOR, label='Others'))
    if include_nodata:
        handles.append(mpatches.Patch(color=NO_DATA_COLOR, label='No data'))
    return handles


In [ ]:
def plot_rank_feature_maps(season_gdfs, rank=1, outfile_stub='Top1Driver'):
    rank_col = f'Top{rank}FeatureID'
    top_ids = get_rank_top8_feature_ids(season_gdfs, rank=rank, top_k=TOP_K)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()
    has_nodata = False
    has_others = False
    for i, season in enumerate(SEASON_CFG.keys()):
        ax = axes[i]
        gdf = season_gdfs[season].copy()
        gdf_plot, plot_colors, _ = apply_top8_fixed_colors(gdf, rank_col, top_ids)
        has_nodata = has_nodata or gdf_plot['FeaturePlotID'].isna().any()
        has_others = has_others or (gdf_plot['FeaturePlotID'] == 'OTHERS').any()
        gdf_plot.plot(color=plot_colors, linewidth=COUNTY_EDGE_WIDTH, edgecolor=COUNTY_EDGE_COLOR, ax=ax)
        states.boundary.plot(ax=ax, color=STATE_EDGE_COLOR, linewidth=STATE_EDGE_WIDTH)
        ax.set_axis_off()
        ax.text(0.02, 1.03, PANEL_LABELS[i], transform=ax.transAxes, ha='left', va='bottom', fontsize=18, fontweight='bold', bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='none', alpha=0.9), clip_on=False)
    handles = build_top8_legend_handles(top_ids=top_ids, include_others=has_others, include_nodata=has_nodata)
    fig.legend(handles=handles, loc='lower center', ncol=5, frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.02))
    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.1, wspace=0.02, hspace=0.04)
    save_plot(fig, outfile_stub)


In [ ]:
def assign_season_dominant_rank_importance(season_gdfs, rank=1):
    id_col = f'SeasonTop{rank}FeatureID'
    name_col = f'SeasonTop{rank}FeatureName'
    imp_col_out = f'SeasonTop{rank}Importance'
    for season in SEASON_CFG.keys():
        gdf = season_gdfs[season]
        rank_ids = season_level_top_raw_feature_ids(gdf, top_n=rank)
        if len(rank_ids) < rank:
            season_gdfs[season][id_col] = None
            season_gdfs[season][name_col] = None
            season_gdfs[season][imp_col_out] = np.nan
            continue
        fid = rank_ids[rank - 1]
        season_gdfs[season][id_col] = fid
        season_gdfs[season][name_col] = pretty_driver_name(fid)
        raw_imp_col = f'Imp_{fid}'
        if raw_imp_col not in gdf.columns:
            season_gdfs[season][imp_col_out] = np.nan
        else:
            season_gdfs[season][imp_col_out] = pd.to_numeric(gdf[raw_imp_col], errors='coerce')
            if '__zero_gain_model__' in gdf:
                season_gdfs[season].loc[gdf['__zero_gain_model__'].eq(True), imp_col_out] = np.nan

def get_global_importance_scale(season_gdfs, max_rank=3):
    valid_vals = []
    for rank in range(1, max_rank + 1):
        imp_col = f'SeasonTop{rank}Importance'
        for season in SEASON_CFG.keys():
            if imp_col in season_gdfs[season].columns:
                arr = pd.to_numeric(season_gdfs[season][imp_col], errors='coerce').values
                arr = arr[np.isfinite(arr)]
                if arr.size > 0:
                    valid_vals.append(arr)
    if len(valid_vals) == 0:
        return (0, 1)
    all_vals = np.concatenate(valid_vals)
    return (float(np.nanmin(all_vals)), float(np.nanmax(all_vals)))

def plot_season_dominant_importance_maps(season_gdfs, rank=1, outfile_stub='Top1Importance', global_vmin=None, global_vmax=None):
    imp_col = f'SeasonTop{rank}Importance'
    name_col = f'SeasonTop{rank}FeatureName'
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.ravel()
    if global_vmin is None or global_vmax is None:
        valid_max = []
        valid_min = []
        for s in SEASON_CFG.keys():
            arr = pd.to_numeric(season_gdfs[s][imp_col], errors='coerce').values
            if np.isfinite(arr).any():
                valid_max.append(np.nanmax(arr))
                valid_min.append(np.nanmin(arr))
        vmax = max(valid_max) if valid_max else 1
        vmin = min(valid_min) if valid_min else 0
    else:
        vmin, vmax = (global_vmin, global_vmax)
    for i, season in enumerate(SEASON_CFG.keys()):
        ax = axes[i]
        gdf = season_gdfs[season]
        gdf.plot(column=imp_col, cmap='YlOrRd', vmin=vmin, vmax=vmax, linewidth=COUNTY_EDGE_WIDTH, edgecolor=COUNTY_EDGE_COLOR, legend=False, ax=ax, missing_kwds={'color': NO_DATA_COLOR, 'edgecolor': COUNTY_EDGE_COLOR})
        states.boundary.plot(ax=ax, color=STATE_EDGE_COLOR, linewidth=STATE_EDGE_WIDTH)
        top_name = gdf[name_col].dropna()
        top_name = top_name.iloc[0] if len(top_name) else 'No data'
        ax.set_title(top_name, fontsize=12, pad=6)
        ax.set_axis_off()
        ax.text(0.02, 1.03, PANEL_LABELS[i], transform=ax.transAxes, ha='left', va='bottom', fontsize=18, fontweight='bold', bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='none', alpha=0.9), clip_on=False)
    fig.subplots_adjust(bottom=0.14, wspace=0.02, hspace=0.04)
    cax = fig.add_axes([0.22, 0.06, 0.56, 0.025])
    sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=Normalize(vmin=vmin, vmax=vmax))
    sm._A = []
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_label('Importance Score', fontsize=12, fontweight='bold')
    cbar.ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    save_plot(fig, outfile_stub)

def get_overall_rank_top_feature_ids(gdf, rank=1, top_k=8):
    rank_col = f'Top{rank}FeatureID'
    vals = pd.Series(gdf[rank_col]).dropna()
    vals = vals[vals.isin(FULL_FEATURE_COLOR_MAP.keys())]
    if vals.empty:
        return []
    return vals.value_counts().index.tolist()[:top_k]


In [ ]:
def plot_overall_maps(merged, gdf, outfile_stub):
    gdf = gdf.copy()
    if '__zero_gain_model__' in gdf:
        gdf.loc[gdf['__zero_gain_model__'].eq(True), [f'Imp_{f}' for f in FIELDS]] = np.nan
    fig = plt.figure(figsize=(24, 15))
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1], hspace=0.1, wspace=0.03)
    labels = ['a', 'b', 'c', 'd', 'e', 'f']
    for i in range(3):
        ax = fig.add_subplot(gs[0, i])
        rank = i + 1
        rank_col = f'Top{rank}FeatureID'
        top_ids = get_overall_rank_top_feature_ids(gdf, rank=rank, top_k=8)
        gdf_plot = gdf.copy()
        gdf_plot['PlotID'] = np.where(gdf_plot[rank_col].isna(), np.nan, np.where(gdf_plot[rank_col].isin(top_ids), gdf_plot[rank_col], 'OTHERS'))
        color_dict = {fid: FULL_FEATURE_COLOR_MAP[fid] for fid in top_ids if fid in FULL_FEATURE_COLOR_MAP}
        plot_colors = gdf_plot['PlotID'].map(color_dict)
        plot_colors = np.where(gdf_plot['PlotID'] == 'OTHERS', OTHERS_COLOR, plot_colors)
        plot_colors = np.where(pd.isna(gdf_plot['PlotID']), NO_DATA_COLOR, plot_colors)
        gdf_plot.plot(color=plot_colors, linewidth=COUNTY_EDGE_WIDTH, edgecolor=COUNTY_EDGE_COLOR, ax=ax)
        states.boundary.plot(ax=ax, color=STATE_EDGE_COLOR, linewidth=STATE_EDGE_WIDTH)
        ax.text(-0.02, 1.02, labels[i], transform=ax.transAxes, fontsize=28, fontweight='bold', va='bottom', ha='right')
        ax.set_axis_off()
        handles = [mpatches.Patch(color=FULL_FEATURE_COLOR_MAP[fid], label=predictors_mapping.get(fid, fid)) for fid in top_ids if fid in FULL_FEATURE_COLOR_MAP]
        if (gdf_plot['PlotID'] == 'OTHERS').any():
            handles.append(mpatches.Patch(color=OTHERS_COLOR, label='Others'))
        if gdf_plot[rank_col].isna().any():
            handles.append(mpatches.Patch(color=NO_DATA_COLOR, label='No data'))
        ax.legend(handles=handles, loc='upper center', ncol=2, fontsize=10, frameon=False, bbox_to_anchor=(0.5, 0.02))
    dominant_ids = overall_top_raw_feature_ids(merged, top_n=3)
    if len(dominant_ids) < 3:
        raise ValueError('Could not identify overall top 3 dominant features from Imp_ columns.')
    all_imp_cols = [f'Imp_{fid}' for fid in dominant_ids]
    global_vmax = 0.0
    for col in all_imp_cols:
        if col in gdf.columns:
            current_max = pd.to_numeric(gdf[col], errors='coerce').max()
            if pd.notna(current_max) and current_max > global_vmax:
                global_vmax = current_max
    if global_vmax <= 0:
        global_vmax = 1.0
    for i in range(3):
        ax = fig.add_subplot(gs[1, i])
        fid = dominant_ids[i]
        imp_col = f'Imp_{fid}'
        gdf.plot(column=imp_col, cmap='YlOrRd', vmin=0, vmax=global_vmax, linewidth=COUNTY_EDGE_WIDTH, edgecolor=COUNTY_EDGE_COLOR, legend=False, ax=ax, missing_kwds={'color': NO_DATA_COLOR, 'edgecolor': COUNTY_EDGE_COLOR})
        states.boundary.plot(ax=ax, color=STATE_EDGE_COLOR, linewidth=STATE_EDGE_WIDTH)
        ax.set_title(predictors_mapping.get(fid, fid), fontsize=15, pad=15)
        ax.text(-0.02, 1.02, labels[i + 3], transform=ax.transAxes, fontsize=28, fontweight='bold', va='bottom', ha='right')
        ax.set_axis_off()
        cax = ax.inset_axes([0.2, -0.08, 0.6, 0.03])
        sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=Normalize(vmin=0, vmax=global_vmax))
        sm._A = []
        cb = fig.colorbar(sm, cax=cax, orientation='horizontal', format='%.2f')
        cb.set_label('Importance Score', fontsize=11, fontweight='bold')
        cb.ax.tick_params(labelsize=10)
    save_plot(fig, outfile_stub)


In [ ]:
gxgb_results = pd.read_csv(OUT / "GXGB.csv", dtype={"GEOID": str}, float_precision="round_trip")
gxgb_results.GEOID = gxgb_results.GEOID.str.zfill(5)
assert set(gxgb_results.model) == set(MODELS)
assert len(gxgb_results) == 2830 * len(MODELS)
CFG["bandwidths"] = {}
frames = {}
for code in MODELS:
    d = gxgb_results.loc[gxgb_results.model.eq(code)].copy()
    assert len(d) == 2830 and d.GEOID.is_unique
    assert set(f"Imp_{feature}" for feature in FEATURES).issubset(d.columns)
    bandwidths = d.bandwidth.unique()
    assert len(bandwidths) == 1 and bandwidths[0] in range(30, 201, 5)
    CFG["bandwidths"][code] = int(bandwidths[0])
    assert np.isfinite(d[["observed", "predicted"]].to_numpy(dtype=float)).all()
    gain = d[[f"Imp_{feature}" for feature in FEATURES]].to_numpy(dtype=float)
    assert np.isfinite(gain).all() and np.all(gain >= 0)
    assert np.all(np.isclose(gain.sum(axis=1), 1, atol=1e-6) | np.isclose(gain.sum(axis=1), 0, atol=1e-8))
    sample = load_data(code)
    assert d.GEOID.tolist() == sample.GEOID.tolist()
    assert np.array_equal(d.observed.to_numpy(), sample.EB_pct.to_numpy())
    frames[code] = compute_ranks(d)
CONTIGUOUS_STATE_FIPS = set("01 04 05 06 08 09 10 11 12 13 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 44 45 46 47 48 49 50 51 53 54 55 56".split())
counties = load_geo("counties")
states = load_geo("states")
counties = counties.loc[counties.STATEFP.isin(CONTIGUOUS_STATE_FIPS)].copy()
states = states.loc[states.STATEFP.isin(CONTIGUOUS_STATE_FIPS)].copy()
counties.GEOID = counties.GEOID.astype(str).str.zfill(5)
counties = counties.to_crs("EPSG:4326").to_crs(MAP_CRS)
states = states.to_crs("EPSG:4326").to_crs(MAP_CRS)
assert not (set(frames["ALL"].GEOID) - set(counties.GEOID))
season_gdfs = {
    code: counties.merge(frames[code], on="GEOID", how="left", validate="1:1")
    for code in SEASONS
}
for rank in [1, 2, 3]:
    assign_season_dominant_rank_importance(season_gdfs, rank=rank)
global_vmin, global_vmax = get_global_importance_scale(season_gdfs, max_rank=3)


In [ ]:
for rank in [1, 2, 3]:
    plot_rank_feature_maps(season_gdfs, rank=rank, outfile_stub=FIGURE_NAMES[f"seasonal_rank{rank}"])
    plot_season_dominant_importance_maps(
        season_gdfs, rank=rank, outfile_stub=FIGURE_NAMES[f"seasonal_mean_rank{rank}"],
        global_vmin=global_vmin, global_vmax=global_vmax
    )
overall = frames["ALL"]
overall_gdf = counties.merge(overall, on="GEOID", how="left", validate="1:1")
plot_overall_maps(overall, overall_gdf, FIGURE_NAMES["overall_top3"])
print([str(plot_folder / f"{stem}.png") for stem in FIGURE_NAMES.values()])
